### Timing: Depending on the system RAM, threads and number of bridges expected to generate, 20 min - 6 h or more

This notebook is used for bridge generation. Bridge has a length of 30 nt.

environment: openFISH_probe  
The main steps of bridge generation includes:  
1. Candidates generation;
2. Unspecific alignment against genome filtering;
3. Unspecific alignment against RO sequences;
4. Orthogonal screening.

## Candidate generation

In [35]:
from seqwalk import design
from Bio.SeqUtils import MeltingTemp
from Levenshtein import distance
from Bio.Seq import reverse_complement
import random

library = design.max_orthogonality(10000, 15, alphabet="ATCG", RCfree=True, GClims=(5, 11))

def get_Tm(seq):
    
    return MeltingTemp.Tm_NN(seq,  dnac1 = 20, check = False,
                            selfcomp = False, Na = 390, Mg = 0.0)

part1 = []
part2 = []
candidates = []

for seq in library:
    if get_Tm(seq) <= 44:
        part1.append(seq)
    elif get_Tm(seq) >= 51:
        part2.append(seq)

fail_count = 0
while len(part1) != 0 and len(part2) != 0:
    c1 = random.choice(part1)
    c2 = random.choice(part2)
    c2_r = reverse_complement(c2)
    if distance(c1, c2) <= 7 and distance(c1, c2_r) <= 7:
        candidates.append(f"{c1}{c2}")
        part1.remove(c1)
        part2.remove(c2)
        fail_count = 0
    else:
        fail_count += 1
        if fail_count >= 50000:
            break

with open("RO_Pad_Design/primary_bridge.fa", "w") as handle:
    for i, seq in enumerate(candidates):
        name = f"bridge_Test_{i}"
        handle.write(f">{name}\n{seq}\n")

Attempting SSM k=6
Attempting SSM k=7
Attempting SSM k=8
Attempting SSM k=9
Number of sequences: 15067
SSM k value: 9


## Unspecific alignment against genome

In [ ]:
import subprocess

call = [
        "~/software/ncbi-blast-2.15.0+/bin/blastn", # Change to your executable blastn binary
        "-query", "RO_Pad_Design/primary_bridge.fa",
        "-db", "/path/to/refseq_rna", # Change the path to blastn refseq_rna database
        "-task", "blastn-short",
        "-evalue", "100",
        "-max_target_seqs", "1000",
        "-strand", "plus", # Aboid being same to RNA
        "-out", "RO_Pad_Design/primary_bridge_mouse.tsv",
        "-outfmt", r'"6 qseqid sseq pident length qstart qend evalue"',
        "-taxids", "10090",
        "-num_threads", "48" # Change threads to an acceptable number
        ]

subprocess.run(" ".join(call), shell = True, check = True)

In [36]:
from tqdm import tqdm

FAIL_LIST = []
with open("RO_Pad_Design/primary_bridge_mouse.tsv", "r") as handle:
    LAST_FAIL = "RANDOM_STRING"
    for line in tqdm(handle):
        qseqid, sseqid, pident, length, start, end = line.strip().split("\t")[0:6]
        if qseqid != LAST_FAIL:
            if float(pident) * int(length) * 0.01 >= 14:
                if int(start) < 7 or int(end) > 7:
                    FAIL_LIST.append(qseqid)
                    LAST_FAIL = qseqid
            else:
                continue
                
j = 0              
with open("RO_Pad_Design/primary_bridge.fa", "r") as handle_in:
    with open("RO_Pad_Design/secondary_bridge.fa", "w") as handle_out:
        lines = handle_in.readlines()
        for i in tqdm(range(0, len(lines), 2)):
            line1 = lines[i].strip()[1:]
            line2 = lines[i+1].strip()
            if j >= len(FAIL_LIST):
                handle_out.write(">" + line1 + "\n")
                handle_out.write(line2 + "\n")
            if line1 != FAIL_LIST[j]:
                handle_out.write(">" + line1 + "\n")
                handle_out.write(line2 + "\n")
            else:
                j += 1

107988it [00:00, 911586.47it/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1233/1233 [00:00<00:00, 817640.61it/s]


## Unspecific alignment against RO

In [38]:
import subprocess

call = [
        "~/software/ncbi-blast-2.15.0+/bin/blastn", # Change to your executable blastn binary
        "-query", "RO_Pad_Design/secondary_bridge.fa",
        "-subject", "RO_Pad_Design/Inclusion_RO.fa",
        "-task", "blastn-short",
        "-evalue", "100",
        "-max_target_seqs", "10000",
        "-strand", "both",
        "-out", "RO_Pad_Design/secondary_bridge_against_RO.tsv",
        "-outfmt", r'"6 qseqid sseqid pident length mismatch"'
        ]

subprocess.run(" ".join(call), shell = True, check = True)

FAIL_LIST = []
with open("RO_Pad_Design/secondary_bridge_against_RO.tsv", "r") as handle:
    LAST_FAIL = "RANDOM_STRING"
    for line in tqdm(handle):
        qseqid, sseqid, pident, length = line.strip().split("\t")[0:4]
        if qseqid != LAST_FAIL:
            if float(pident) * int(length) * 0.01 >= 8:
                    FAIL_LIST.append(qseqid)
                    LAST_FAIL = qseqid
            else:
                continue

j = 0              
with open("RO_Pad_Design/secondary_bridge.fa", "r") as handle_in:
    with open("RO_Pad_Design/third_bridge.fa", "w") as handle_out:
        lines = handle_in.readlines()
        for i in tqdm(range(0, len(lines), 2)):
            line1 = lines[i].strip()[1:]
            line2 = lines[i+1].strip()
            if j >= len(FAIL_LIST):
                handle_out.write(">" + line1 + "\n")
                handle_out.write(line2 + "\n")
            elif line1 != FAIL_LIST[j]:
                handle_out.write(">" + line1 + "\n")
                handle_out.write(line2 + "\n")
            else:
                j += 1

468it [00:00, 627737.22it/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 104/104 [00:00<00:00, 550767.19it/s]


## Orthogonal screening

In [41]:
# Optional: filter out candidates similary to exisiting bridges
# ExistingBridge_list = [
#     'TGTGGCACATTATCTTAAAGGCTCTCGAAA',
#     'GACTCTGCGAACATATTGGTGCCGACATCA',
#     'TAGTCCGTGTGTCAGAATACCTAGCGGAAT'
# ]

# with open("RO_Pad_Design/Existing_bridge.fa", "w") as handle:
#     for i, seq in enumerate(ExistingBridge_list):
#         handle.write(f">Existing_{i}\n{seq}\n")

# call = [
#         "~/software/ncbi-blast-2.15.0+/bin/blastn", # Change to your executable blastn binary
#         "-query", "RO_Pad_Design/third_bridge.fa",
#         "-subject", "RO_Pad_Design/Existing_bridge.fa",
#         "-task", "blastn-short",
#         "-evalue", "100",
#         "-max_target_seqs", "10000",
#         "-strand", "both",
#         "-out", "RO_Pad_Design/Orthogonal_Existing.tsv",
#         "-outfmt", r'"6 qseqid sseqid pident length qstart qend sstart send"'
#         ]

# subprocess.run(" ".join(call), shell = True, check = True)

# FAIL_LIST = []
# with open("RO_Pad_Design/Orthogonal_Existing.tsv", "r") as handle:
#     LAST_FAIL = "RANDOM_STRING"
#     for line in tqdm(handle):
#         qseqid, sseqid, pident, length, qstart, qend, sstart, send = line.strip().split("\t")[0:8]
#         if qseqid != LAST_FAIL:
#             if float(pident) * int(length) * 0.01 >= 8:
#                 if int(qstart) < 6 or int(qend) < 6 or int(sstart) < 6 or int(send) < 6:
#                     FAIL_LIST.append(qseqid)
#                     LAST_FAIL = qseqid
#             elif float(pident) * int(length) * 0.01 >= 14:
#                 FAIL_LIST.append(qseqid)
#                 LAST_FAIL = qseqid
#             else:
#                 continue

# j = 0              
# with open("RO_Pad_Design/third_bridge.fa", "r") as handle_in:
#     with open("RO_Pad_Design/fourth_bridge.fa", "w") as handle_out:
#         lines = handle_in.readlines()
#         for i in tqdm(range(0, len(lines), 2)):
#             line1 = lines[i].strip()[1:]
#             line2 = lines[i+1].strip()
#             if j >= len(FAIL_LIST):
#                 handle_out.write(">" + line1 + "\n")
#                 handle_out.write(line2 + "\n")
#             elif line1 != FAIL_LIST[j]:
#                 handle_out.write(">" + line1 + "\n")
#                 handle_out.write(line2 + "\n")
#             else:
#                 j += 1

12it [00:00, 68200.07it/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 37/37 [00:00<00:00, 323985.90it/s]


In [42]:
call = [
        "~/software/ncbi-blast-2.15.0+/bin/blastn", # Change to your executable blastn binary
        "-query", "RO_Pad_Design/third_bridge.fa", # input RO_Pad_Design/fourth_bridge.fa if you filtered against existing bridge
        "-subject", "RO_Pad_Design/third_bridge.fa", # input RO_Pad_Design/fourth_bridge.fa if you filtered against existing bridge
        "-task", "blastn-short",
        "-evalue", "100",
        "-max_target_seqs", "10000",
        "-strand", "both",
        "-out", "RO_Pad_Design/Orthogonal_DeLOB.tsv",
        "-outfmt", r'"6 qseqid sseqid pident length qstart qend sstart send"'
        ]

subprocess.run(" ".join(call), shell = True, check = True)

CompletedProcess(args='~/software/ncbi-blast-2.15.0+/bin/blastn -query RO_Pad_Design/third_bridge.fa -subject RO_Pad_Design/third_bridge.fa -task blastn-short -evalue 100 -max_target_seqs 10000 -strand both -out RO_Pad_Design/Orthogonal_DeLOB.tsv -outfmt "6 qseqid sseqid pident length qstart qend sstart send"', returncode=0)

In [46]:
import networkx as nx
import random
G = nx.Graph()

with open("RO_Pad_Design/third_bridge.fa", "r") as handle: # input RO_Pad_Design/third_RO.fa if you filtered against existing ROs
    lines = handle.readlines()
    for i in tqdm(range(0, len(lines), 2)):
        line1 = lines[i].strip()[1:]
        line2 = lines[i+1].strip()
        G.add_node(line1, seq=line2)

with open("RO_Pad_Design/Orthogonal_DeLOB.tsv", "r") as handle:
    for line in tqdm(handle):
        qseqid, sseqid, pident, length, qstart, qend, sstart, send = line.strip().split("\t")[0:8]
        if qseqid != sseqid:
            if float(pident) * int(length) * 0.01 >= 8:
                if int(qstart) < 6 or int(qend) < 6 or int(sstart) < 6 or int(send) < 6:
                    G.add_edge(qseqid, sseqid)
            elif float(pident) * int(length) * 0.01 >= 14:
                G.add_edge(qseqid, sseqid)
            else:
                continue

node_list = list(G.nodes)

with open("RO_Pad_Design/Inclusion_bridge.fa", 'w') as In_handle, open("RO_Pad_Design/Exclusion_bridge.fa", 'w') as Ex_handle:
    while len(node_list) > 0:
        
        random_node = random.choice(node_list)
        edged_nodes = list(G[random_node])
        seq = G.nodes[random_node]["seq"]
        In_handle.write(f">{random_node}\n{seq}\n")
        G.remove_node(random_node)
        for edge_n in edged_nodes:
            seq = G.nodes[edge_n]["seq"]
            Ex_handle.write(f">{edge_n}\n{seq}\n")
            G.remove_node(edge_n)
        
        node_list = list(G.nodes)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 37/37 [00:00<00:00, 229910.00it/s]
183it [00:00, 253905.93it/s]


<div class="alert alert-block alert-warning">
<b>RO_Pad_Design/Inclusion_bridge.fa now can be used for folllowing pad join</b>
</div>